## Unit 09. ODE 常微分方程式求解 課堂作業
- 課程名稱: 電腦在化工上之應用 (ChemE 3502)
- 課程編號: CHEXXXX
- 學年度: 114下
- 上課時間: 每週四 09:00-12:00
-
- 指導教授: 莊曜禎 助理教授
- 學生姓名: ＯＯＯ
- 學號: m12345678
- email帳號: fcu.m12345678@gmail.com

---
### 環境設定

In [ ]:
from pathlib import Path
import os

# ========================================
# 路徑設定 (兼容 Colab 與 Local)
# ========================================
UNIT_OUTPUT_DIR = 'Unit09_HW'

try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ 偵測到 Colab 環境，準備掛載 Google Drive...")
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    IN_COLAB = False
    print("✓ 偵測到 Local 環境")

try:
    shortcut_path = '/content/ChemE-3502'
    os.remove(shortcut_path)
except (FileNotFoundError, OSError):
    pass

if IN_COLAB:
    source_path = Path('/content/drive/My Drive/Colab Notebooks/ChemE-3502')
    os.symlink(source_path, shortcut_path)
    shortcut_path = Path(shortcut_path)
    if source_path.exists():
        NOTEBOOK_DIR = shortcut_path / 'Unit09'
        OUTPUT_DIR   = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
        FIG_DIR      = OUTPUT_DIR / 'figs'
    else:
        print("⚠️ 找不到雲端 ChemE-3502 路徑，請確認資料夾名稱是否正確")
else:
    NOTEBOOK_DIR = Path.cwd()
    OUTPUT_DIR   = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
    FIG_DIR      = OUTPUT_DIR / 'figs'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Notebook 工作目錄: {NOTEBOOK_DIR}")
print(f"✓ 結果輸出目錄:     {OUTPUT_DIR}")
print(f"✓ 圖檔輸出目錄:     {FIG_DIR}")

---
### 載入套件

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp, solve_bvp
import warnings
warnings.filterwarnings('ignore')

# 繪圖樣式設定
plt.rcParams.update({
    'figure.dpi'    : 100,
    'axes.grid'     : True,
    'grid.alpha'    : 0.3,
    'font.size'     : 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'lines.linewidth': 2,
    'axes.unicode_minus': False,
})

print("✓ 套件載入完成")
print(f"  numpy  版本: {np.__version__}")
import scipy
print(f"  scipy  版本: {scipy.__version__}")
import matplotlib
print(f"  matplotlib 版本: {matplotlib.__version__}")

---
### **I. 練習題：串聯式三加熱槽系統之動態模擬 (IVP)**

考慮如下圖所示之串聯式三個加熱槽系統（習題 4-7-13）。

假設各槽混合良好，槽內溫度均一。各槽之**能量平衡方程式**為：

$$
M C_P \frac{d T_i}{d t} = W C_P T_{i-1} + UA(T_S - T_i) - W C_P T_i, \quad i = 1, 2, 3
$$

其中 $T_0$ 為進料溫度（已知），化簡後得：

$$
\frac{d T_i}{d t} = \frac{W C_P}{M C_P}(T_{i-1} - T_i) + \frac{UA}{M C_P}(T_S - T_i), \quad i = 1,2,3
$$

**系統參數：**

| 參數 | 符號 | 數值 | 單位 |
|------|------|------|------|
| 各槽質量 | $M$ | 1000 | kg |
| 熱容量 | $C_P$ | 2.0 | kJ/(kg·°C) |
| 進料流率 | $W$ | 100 | kg/min |
| 熱傳係數×面積 | $UA$ | 10 | kJ/(min·°C) |
| 加熱蒸氣溫度 | $T_S$ | 250 | °C |
| 進料溫度（ $T_0$ ）| $T_0$ | 20 | °C |
| 各槽初溫 | $T_i(0)$ | 20 | °C |

**模擬時間**： $0 \leq t \leq 200 \, \text{min}$

**參考答案（ $t \to \infty$ ）**：各槽終溫分別為 **30.952、41.380 及 51.295 °C**

### **1. 建立 ODE 系統與設定參數**
- 依題目公式，寫出三槽聯立 ODE 函式 `tank_odes(t, T)`
- 設定系統參數 `M, Cp, W, UA, Ts, T0`
- 設定起始條件 `T0_init = [20, 20, 20]`
- 設定模擬時間 `t_span = (0, 200)`，輸出點 `t_eval = np.linspace(0, 200, 1000)`

### **2. 使用 `solve_ivp` 求解 ODE 系統**
- 使用 `scipy.integrate.solve_ivp()` 搭配 `method='RK45'` 求解
- 確認 `sol.success == True`
- 取出各槽溫度解 `T1, T2, T3 = sol.y`
- 印出 $t = 200$ min 時的各槽溫度，與參考答案做比較

### **3. 繪製各槽溫度隨時間之變化圖**
- 在同一張圖上繪製 Tank 1、2、3 之溫度對時間曲線
- 以不同顏色與線型標示各槽，並加入圖例
- 標示水平虛線表示各槽的穩態溫度參考值（30.952、41.380、51.295 °C）
- x 軸：Time (min)，y 軸：Temperature (°C)
- 儲存圖檔至 `FIG_DIR / 'fig01_tank_temp.png'`

### **4. 確認穩態值並與解析解比較**
- 從 ODE 解中取出 $t=200$ min 的溫度值，確認已達穩態
- 另外由穩態方程式（令 $dT_i/dt = 0$ ）推導出各槽的**解析解**，計算如下：

$$
T_i^* = \frac{W C_P\, T_{i-1}^* + UA\, T_S}{W C_P + UA}, \quad i = 1, 2, 3
$$

- 以程式碼計算各槽解析穩態值，並與數值解比較誤差（應 $< 0.01$ °C）
- 印出比較表格

### **5. 敏感度分析：改變熱傳係數 UA**
- 固定其他參數，改變 $UA \in \{5, 10, 20, 50\}$ kJ/(min·°C)，分別求解 ODE
- 在同一張圖上繪製第三槽（Tank 3）溫度對時間曲線以比較不同 UA 的影響
- 製作表格，列出各 UA 值對應的三槽穩態溫度
- **分析**：UA 增大時，系統動態如何改變？穩態溫度有何趨勢？
- 儲存圖檔至 `FIG_DIR / 'fig02_UA_sensitivity.png'`

### 💭 思考題

1. 為什麼三個加熱槽的穩態溫度不一樣？各槽的升溫速度有何差異？請從能量平衡的角度解釋。
2. 若將槽的質量 $M$ 加倍，系統到達穩態的時間會如何改變？穩態溫度是否也會改變？
3. 若串聯槽數從 3 個增加到 5 個，且參數不變，最末槽的穩態溫度是否會比 3 槽系統更高還是更低？為什麼？
4. 若進料溫度 $T_0$ 由 20°C 改為 40°C，三槽的穩態溫度會如何變化（定性分析）？
5. 本題屬於「線性 IVP」，其解析解可以矩陣指數法求得。若改為非線性 ODE（如反應槽），數值解法的優勢為何？

---
## II. 額外加分作業（自由繳交）
- 學生可自由選擇是否繳交加分作業（下週上課前完成 email 電子檔）
- 分數會加在最後總成績上，每份作業加 0.1 ~ 1.0 分
- 分數加的不多，真的不一定要交，正常出席上課做好期末報告即可
- 加分作業不一定要全部做完，想繳交至少完成其中一項即可
- 請務必自行完成！你可以問 AI、問同學、問學長姊，但一定要自行「吸收」「執行」「整理」結果
- 不要貼 AI 的答案寄給老師看，你自己看就好
- 如果系統自動比對發現作業內容（與他人雷同、原創性低、抄襲比例過高），作業分數會 0 分

---
### 加分作業：圓錐形散熱片之溫度分布（BVP）

考慮一個圓錐形散熱片（Conical Fin），假設 $R$ 方向溫度均一， $x$ 軸方向之一維穩態熱傳可以下列二階 ODE 描述（習題 4-7-8）：

$$
\frac{d^2\theta}{d\tau^2} - \frac{2\tan\alpha}{\eta_0 - \tau\tan\alpha}\frac{d\theta}{d\tau} - \frac{2\beta}{\cos\alpha\,(\eta_0 - \tau\tan\alpha)}\,\theta = 0 \tag{1}
$$

**邊界條件：**

$$
\theta\big|_{\tau=0} = 1 \quad \text{（底部，固定牆面溫度）}
$$

$$
\left.\frac{d\theta}{d\tau}\right|_{\tau=1} = -\beta\,\theta\big|_{\tau=1} \quad \text{（頂部，自然對流散熱）}
$$

其中無因次變數定義為： $\theta = T/T_0$ 、 $\tau = x/L$ 、 $\eta_0 = R/L$ 、 $\beta = hL/k$

**系統參數：**

| 參數 | 符號 | 數值 |
|------|------|------|
| 半錐角正切值 | $\tan\alpha$ | 0.08 |
| 無因次頂端半徑 | $\eta_0$ | 0.1 |
| Biot 數 | $\beta$ | 0.5 |

**注意**：方程式在 $\tau=0$ 附近奇異（分母中有 $\eta_0 - \tau\tan\alpha$ ），需注意初始網格點不包含 $\tau=0$ 或使用足夠密的網格。

**熱效率（Fin Efficiency）：**

$$
\eta_{\text{fin}} = -\frac{1}{\beta}\left.\frac{d\theta}{d\tau}\right|_{\tau=0}
$$

**參考答案**：熱效率 $\eta_{\text{fin}} = 5.1777$

### **A. 轉換為一階 ODE 系統**
令 $y_0 = \theta$ 、 $y_1 = d\theta/d\tau$ ，將式 (1) 化為一階聯立系統，並寫出 `fin_ode(tau, y)` 函式。
邊界條件函式 `fin_bc(ya, yb)` 應回傳長度為 2 的殘差陣列：

$$
\text{bc}[0] = y_0\big|_{\tau=0} - 1, \qquad \text{bc}[1] = y_1\big|_{\tau=1} + \beta\, y_0\big|_{\tau=1}
$$

設定初始網格 `tau_init = np.linspace(0, 1, 100)` 及線性初始猜測（ $\theta$ 由 1 線性降至 0.5）。

### **B. 使用 `solve_bvp` 求解溫度分布**
- 呼叫 `solve_bvp(fin_ode, fin_bc, tau_init, y_init, tol=1e-5, max_nodes=5000)`
- 確認 `sol.success == True`，印出求解狀態與最大殘差
- 取出密集解曲線 `tau_dense = np.linspace(0, 1, 300)` 的溫度值 `theta_sol = sol.sol(tau_dense)[0]`

### **C. 繪製散熱片溫度分布圖**
- 繪製 $\theta(\tau)$ 對 $\tau$ 的曲線（x 軸：Dimensionless position τ，y 軸：Dimensionless temperature θ）
- 標示 $\theta(0) = 1$ （底部邊界）及頂端溫度 $\theta(1)$ 的數値
- 儲存圖檔至 `FIG_DIR / 'fig03_conical_fin.png'`

### **D. 計算散熱片熱效率**
- 由數值解取出 $d\theta/d\tau\big|_{\tau=0}$ （即 `sol.sol(np.array([0.0]))[1][0]` ）
- 依公式計算熱效率：

$$
\eta_{\text{fin}} = -\frac{1}{\beta}\left.\frac{d\theta}{d\tau}\right|_{\tau=0}
$$

- 印出計算結果，與參考答案 $\eta_{\text{fin}} = 5.1777$ 比較，計算相對誤差（應 $< 0.1\%$ ）
- **分析**：此圓錐形散熱片的熱效率 $> 1$ ，在物理意義上代表什麼？

---
# 想對老師說的話